## 1. 核心思想
在文档进入向量库**之前**，通过优化索引结构和分块策略，从源头提升检索质量。解决"分块粒度"与"上下文完整性"之间的矛盾。

## 2. 核心逻辑伪代码

In [ ]:
class AutoMergingRetriever:
    def __init__(self):
        self.doc_store = {} # 存储父块内容 (Key: ParentID, Value: Text)
        self.threshold = 0.5 # 设定合并阈值 (例如：命中子块数占父块总子块的比例，或绝对数量)

    def index_document(self, big_doc_text):
        """索引阶段：切分父子块"""
        parent_id = generate_uuid()

        # 1. 存父块 (不向量化，只存文本)
        self.doc_store[parent_id] = big_doc_text

        # 2. 切分子块
        small_chunks = split_text(big_doc_text, size=128)

        # 3. 子块向量化并存储映射
        for chunk in small_chunks:
            vector = embedding_model.encode(chunk)
            # 关键：在子块元数据中记录 parent_id
            vector_db.insert(vector, metadata={"parent_id": parent_id, "is_child": True})

    def retrieve(self, query):
        """检索阶段：自动合并逻辑"""
        # 1. 检索子块
        child_hits = vector_db.search(query, top_k=5)

        # 2. 统计父块命中情况
        parent_stats = {} # {parent_id: count}
        for hit in child_hits:
            pid = hit.metadata["parent_id"]
            parent_stats[pid] = parent_stats.get(pid, 0) + 1

        final_results = []

        # 3. 决策：返回子块还是父块？
        for pid, count in parent_stats.items():
            # 逻辑：如果同一个父块下有多个子块被命中 (例如 >= 2)
            if count >= 2:
                # 升级：提取完整的父块
                final_results.append(self.doc_store[pid])
            else:
                # 保持：只返回命中的那个子块 (需要从 hits 里找回对应的子块文本)
                child_text = find_child_text_in_hits(child_hits, pid)
                final_results.append(child_text)

        return final_results

## 3. 完整实现（Milvus + OpenAI）
我们需要构建两个存储：

Vector Store (Milvus): 存储子块 (Child Chunks) 的向量。

Doc Store (内存/KV): 存储父块 (Parent Chunks) 的完整文本（模拟 MongoDB/Redis）

### 3.1 配置

In [13]:
import os
import uuid
from pymilvus import MilvusClient, DataType
from openai import OpenAI
import numpy as np

# --- 配置部分 ---
os.environ["OPENAI_API_KEY"] = "sk-OdqRypFlfJLKYvLmV6GsG9j0u6CRFBYKErn4xV1Wm0R3q0y9"  # 替换你的 Key
os.environ["OPENAI_BASE_URL"] = "https://api.openai-proxy.org/v1"
client = OpenAI()
milvus_client = MilvusClient(uri="http://111.228.53.183:19530")

COLLECTION_NAME = "child_chunks_index"
DIMENSION = 3072
MERGE_THRESHOLD = 2  # 核心参数：如果一个父块中有 >= 2 个子块被命中，就返回父块

# --- 模拟 KV 存储 (实际生产通常用 Redis/MongoDB) ---
# 存储结构: { "parent_id_xxx": "完整的长文本内容..." }
PARENT_DOC_STORE = {}

### 3.2 核心函数封装

In [14]:
def get_embedding(text):
    response = client.embeddings.create(input=text, model="text-embedding-3-large")
    return response.data[0].embedding

def simple_text_splitter(text, chunk_size=100, overlap=20):
    """
    简单的滑动窗口切分器 (模拟子块切分)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += (chunk_size - overlap)
    return chunks

### 3.3 准备milvus集合

In [15]:
# --- Step 1: 初始化 Milvus (仅存子块) ---

if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=DIMENSION,
    auto_id=True,
    enable_dynamic_field=True
)
print("索引初始化完成 (Child Vector Store)。\n")

索引初始化完成 (Child Vector Store)。



### 3.4 构建索引

In [16]:
# --- Step 2: 索引构建 (Parent -> Children -> Index) ---

# 模拟原始文档 (父块)
# 这是一个关于 Python 异常处理的完整段落
documents = [
    """Python 中的异常处理使用 try-except 语句。try 块中包含可能引发异常的代码。except 块捕获并处理特定异常。finally 块无论是否发生异常都会执行，常用于资源释放。else 块在未发生异常时执行。自定义异常可以通过继承 Exception 类来实现。""",

    """Python 的列表推导式提供了一种简洁的创建列表的方法。它由括号内的表达式和 for 子句组成。列表推导式比循环更高效，代码更易读。例如 [x*2 for x in range(10)] 生成偶数列表。它也支持 if 条件过滤。"""
]

print("正在处理文档...")

for doc_text in documents:
    # 1. 生成父块 ID 并存入 KV Store
    parent_id = str(uuid.uuid4())
    PARENT_DOC_STORE[parent_id] = doc_text

    # 2. 将父块切分为细粒度的子块 (例如每块 50-100 字)
    child_chunks = simple_text_splitter(doc_text, chunk_size=50, overlap=10)

    print(f" -> 父块 (ID: {parent_id[:8]}...) 被切分为 {len(child_chunks)} 个子块")

    # 3. 子块向量化并入库
    data_rows = []
    for child_text in child_chunks:
        vec = get_embedding(child_text)
        data_rows.append({
            "vector": vec,
            "text": child_text,      # 子块文本
            "parent_id": parent_id   # 关键：关联父块
        })


    # 4. 将某个父块下的所有子块存储到milvus
    insert_result=milvus_client.insert(collection_name=COLLECTION_NAME, data=data_rows)



正在处理文档...
 -> 父块 (ID: e7b4ffd6...) 被切分为 4 个子块
 -> 父块 (ID: 167d1860...) 被切分为 3 个子块


## 3.5. 检索
- 定义检索函数

In [17]:
# --- Step 3: 检索与自动合并逻辑 ---

def auto_merging_retrieval(query):
    print(f"用户查询: {query}")

    # 1. 向量检索 (检索子块)
    query_vec = get_embedding(query)

    # 检索 Top 5 个子块
    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_vec],
        limit=5,
        output_fields=["text", "parent_id"]
    )

    # 2. 统计与聚合
    # 结构: { parent_id: [child_hit_1, child_hit_2] }
    hit_grouping = {}

    print(" -> 命中子块详情:")
    for hit in results[0]:
        pid = hit["entity"]["parent_id"]
        child_text = hit["entity"]["text"]
        score = hit["distance"]

        if pid not in hit_grouping:
            hit_grouping[pid] = []
        hit_grouping[pid].append(child_text)

        print(f"    - [父ID:{pid[:6]}] 子块: {child_text[:20]}... (分: {score:.4f})")

    # 3. 决策逻辑 (Merge Logic)
    final_context = []

    print("\n -> 合并决策执行中...")
    for pid, children in hit_grouping.items():
        count = len(children)

        if count >= MERGE_THRESHOLD:
            # 触发合并：放弃子块，返回父块
            parent_text = PARENT_DOC_STORE.get(pid)
            if parent_text:
                final_context.append(f"【完整父块】{parent_text}")
                print(f"    [√] 父块 {pid[:6]} 命中 {count} 次 (>=阈值 {MERGE_THRESHOLD}) -> 自动合并返回完整父块")
        else:
            # 未触发合并：仅返回命中的子块
            # 为了防止上下文重复，如果父块已经因为其他原因被加入了，这里就不需要加子块了(简单起见这里不做去重)
            for child in children:
                final_context.append(f"【子块片段】{child}")
                print(f"    [x] 父块 {pid[:6]} 命中 {count} 次 (<阈值 {MERGE_THRESHOLD}) -> 仅返回子块")

    return final_context


def generate_answer_with_context(query, context_list):
    """
    将合并后的上下文（可能是完整的父块，也可能是零散的子块）喂给 LLM
    """
    if not context_list:
        return "抱歉，知识库中未找到相关内容。"

    # 将列表拼接为字符串
    # 提示：在这里可以看到，如果触发了自动合并，context_str 将包含逻辑完整的长文本
    context_str = "\n\n".join(context_list)

    prompt = f"""
    你是一个专业的 Python 技术专家。请基于以下提供的【参考文档】回答用户的【问题】。

    【参考文档】:
    {context_str}

    【用户问题】:
    {query}

    要求：
    1. 如果参考文档提供了完整的代码逻辑（如 try-except-finally），请在回答中完整解释。
    2. 引用文档中的具体概念。
    """

    # 调用 OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini", # 推荐使用长窗口模型，因为父块可能很长
        messages=[
            {"role": "system", "content": "你是一个有用的助手。"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3 # 降低随机性，基于文档回答
    )

    return response.choices[0].message.content

## 3.7 检索
- 测试

In [18]:
print("开始执行完整 RAG 流程 (Auto-merging)")
# 1. 用户提问
# 这个问题的特点是：它跨越了文档的多个部分
# "try" 和 "except" 可能在一个子块，"finally" 和 "else" 可能在另一个子块
user_query = "请详细解释 Python 异常处理中 try, except, else, 和 finally 各自的作用及执行顺序。"

# 2. 检索 + 自动合并 (Retrieve & Merge)
# 这里会执行之前的逻辑：
# - 检索到子块 A (try/except)
# - 检索到子块 B (finally/else)
# - 发现它们属于同一个 Parent -> 自动合并 -> 返回整个 Parent
merged_context = auto_merging_retrieval(user_query)

print(f"\n>> 检索系统返回了 {len(merged_context)} 个上下文片段。")

# 3. LLM 生成 (Generate)
print("\n>> 正在请求 LLM 生成最终答案...\n")
final_answer = generate_answer_with_context(user_query, merged_context)

# 4. 输出结果
print("-" * 20 + " 最终 RAG 回答 " + "-" * 20)
print(final_answer)

开始执行完整 RAG 流程 (Auto-merging)
用户查询: 请详细解释 Python 异常处理中 try, except, else, 和 finally 各自的作用及执行顺序。
 -> 命中子块详情:
    - [父ID:e7b4ff] 子块: Python 中的异常处理使用 try-... (分: 0.5954)
    - [父ID:e7b4ff] 子块: 引发异常的代码。except 块捕获并处... (分: 0.5554)
    - [父ID:e7b4ff] 子块: 生异常都会执行，常用于资源释放。else... (分: 0.4772)
    - [父ID:e7b4ff] 子块: 过继承 Exception 类来实现。... (分: 0.3261)
    - [父ID:167d18] 子块: Python 的列表推导式提供了一种简洁... (分: 0.2772)

 -> 合并决策执行中...
    [√] 父块 e7b4ff 命中 4 次 (>=阈值 2) -> 自动合并返回完整父块
    [x] 父块 167d18 命中 1 次 (<阈值 2) -> 仅返回子块

>> 检索系统返回了 2 个上下文片段。

>> 正在请求 LLM 生成最终答案...

-------------------- 最终 RAG 回答 --------------------
在 Python 中，异常处理是通过 `try-except` 语句实现的，主要用于捕获和处理可能引发的异常。以下是对 `try`、`except`、`else` 和 `finally` 各自作用及执行顺序的详细解释：

1. **try 块**：
   - `try` 块中包含可能引发异常的代码。当程序执行到 `try` 块时，如果没有发生任何异常，程序会继续执行 `try` 块后面的代码。
   - 如果 `try` 块中的代码引发了异常，程序会立即停止执行 `try` 块中的剩余代码，并转向相应的 `except` 块。

2. **except 块**：
   - `except` 块用于捕获和处理在 `try` 块中引发的特定异常。可以根据需要定义多个 `except` 块来处理不同类型的异常。
   - 如果 `try` 块中的代码引发了异常，程序会查找与该异常匹配的